# Transformada de Hough Estándar (SHT)

El principio fundamental es la conversión de puntos de un espacio cartesiano $(x, y)$ a un espacio de parámetros, permitiendo que el problema de detectar una línea se convierta en un problema de encontrar intersecciones en un acumulador.

**El problema de la pendiente infinita:**

En la ecuación de la recta tradicional $y = mx + c$, una línea vertical tiene una pendiente $m$ infinita, lo que impide su representación computacional. Por ello, se utiliza la notación polar:$$\rho = x \cos \theta + y \sin \theta$$

Donde:

- $\rho$ (rho): Es la distancia perpendicular desde el origen hasta la línea.

- $\theta$ (theta): Es el ángulo formado por esa perpendicular con el eje horizontal.

## El Espacio de Parámetros (Hough Space)

- Punto a Sinusoide: 

   Un punto único en la imagen $(x, y)$ se transforma en una curva sinusoidal en el espacio $(\rho, \theta)$.

- Línea a Punto: 

   Si varios puntos en la imagen pertenecen a la misma recta, todas sus sinusoides en el espacio de Hough se cruzarán en un único punto común $(\rho, \theta)$.

**El Sistema de Votación**

Para implementar esto, el algoritmo crea una matriz bidimensional llamada acumulador.

1. Se discretiza el espacio de $(\rho, \theta)$ en celdas.

2. Para cada píxel de borde detectado (por ejemplo, mediante Canny), se calculan todos los posibles valores de $\rho$ para distintos ángulos $\theta$.

3. Se suma un "voto" a cada celda $(\rho, \theta)$ correspondiente.

4. Las celdas con la mayor cantidad de votos representan las líneas más probables de la imagen original.

In [ ]:
# ==========================================
# 4.3. HOUGH TRANSFORM 
# ==========================================

# 1. Transformada de Hough
lineas_hough = cv2.HoughLinesP(img_roi, rho=1, theta=np.pi/180, threshold=20, 
                               minLineLength=20, maxLineGap=300)

img_hough_bruto = np.copy(img_dia_rgb)
img_resultado_final = np.copy(img_dia_rgb)

pendientes_izq, interceptos_izq = [], []
pendientes_der, interceptos_der = [], []

if lineas_hough is not None:
    for linea in lineas_hough:
        x1, y1, x2, y2 = linea[0]
        
        
        cv2.line(img_hough_bruto, (x1, y1), (x2, y2), (255, 0, 0), 2)
        
        if x1 == x2: continue # Evitar divisiones por cero
            
        
        slope = (y2 - y1) / (x2 - x1)
        intercept = y1 - slope * x1
        
        # Filtrado de líneas horizontales y clasificación en izquierda/derecha
        if slope < -0.5:
            pendientes_izq.append(slope)
            interceptos_izq.append(intercept)
        elif slope > 0.5:
            pendientes_der.append(slope)
            interceptos_der.append(intercept)

rho=1 y theta=np.pi/180: Definen la resolución del acumulador en el espacio de Hough.

threshold=20: El número mínimo de "votos" o intersecciones para que un conjunto de puntos se considere una línea.

minLineLength=20: Descarta cualquier segmento más corto de 20 píxeles, eliminando ruidos pequeños.

maxLineGap=300: Permite unir segmentos que están separados por hasta 300 píxeles, lo cual es vital para detectar líneas de carril discontinuas.

Dentro del bucle for, el código calcula la geometría de cada segmento detectado usando las Ecuaciones 15 y 16 del artículo:

- **Pendiente (slope)**: Se calcula como $m = \frac{y_2 - y_1}{x_2 - x_1}$.

- **Intersección (intercept)**: Se calcula como $c = y_1 - m \cdot x_1$.

**Lógica de filtrado:**

- slope < -0.5: Se clasifica como carril izquierdo. En coordenadas de imagen (donde el origen está arriba a la izquierda), una pendiente negativa indica una línea que se inclina hacia la derecha según sube hacia el horizonte.

- slope > 0.5: Se clasifica como carril derecho.Filtro horizontal: Las líneas con pendientes entre -0.5 y 0.5 se ignoran, ya que suelen ser ruidos del horizonte o sombras transversales.

## Lane Line Averaging

La Transformada de Hough suele detectar decenas de pequeños segmentos para un solo carril debido a discontinuidades en la pintura, sombras o texturas del asfalto. Si dibujáramos todos estos segmentos directamente, la visualización sería caótica y el sistema de control del vehículo recibiría señales contradictorias.

In [ ]:
# 3. Promediado (Lane Line Averaging)
y_max = img_resultado_final.shape[0]
y_min = int(y_max * 0.6)

def dibujar_carril_promediado(pendientes, interceptos, color):
    if len(pendientes) > 0:
        # Calculamos el average slope e intercept
        avg_slope = np.mean(pendientes)
        avg_intercept = np.mean(interceptos)
        
        # Invertimos la fórmula para hallar las X finales: x = (y - intercept) / slope
        x_fondo = int((y_min - avg_intercept) / avg_slope)
        x_base = int((y_max - avg_intercept) / avg_slope)
        
        cv2.line(img_resultado_final, (x_fondo, y_min), (x_base, y_max), color, 8)

# Dibujamos las líneas promediadas finales
dibujar_carril_promediado(pendientes_izq, interceptos_izq, (0, 255, 0)) # Verde
dibujar_carril_promediado(pendientes_der, interceptos_der, (0, 255, 0)) # Verde

**El Promediado (The Mean)**

Una vez agrupados, se calcula el promedio aritmético de todas las pendientes e interceptos de cada grupo:
$$m_{promedio} = \text{mean}(p_{izq} \text{ o } p_{der})$$
$$c_{promedio} = \text{mean}(c_{izq} \text{ o } c_{der})$$

Este paso es vital porque actúa como un filtro de paso bajo, eliminando el impacto de segmentos "aberrantes" causados por ruido o sombras que tengan pendientes muy diferentes a la media del carril.5. 

**Extrapolación y Dibujo**

Con un solo par de valores $(m, c)$ para cada lado, el sistema puede dibujar líneas continuas que van desde la base de la imagen ($y_{max}$) hasta el horizonte definido ($y_{min}$).

Para encontrar la coordenada $x$ en el horizonte o en la base, se despeja la ecuación: $x = \frac{y - c}{m}$.